In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

import pymedphys
from pymedphys._experimental.autosegmentation import filtering as _filtering
from pymedphys._experimental.autosegmentation import indexing as _indexing
from pymedphys._experimental.autosegmentation import mask as _mask
from pymedphys._experimental.autosegmentation import pipeline as _pipeline

In [ ]:
(
    _,
    _,
    _,
    ct_uid_to_structure_uid,
    structure_uid_to_ct_uids,
    _,
    structure_names_by_ct_uid,
    structure_names_by_structure_set_uid,
    _,
    _,
) = _pipeline.get_dataset_metadata()

In [ ]:
structure_names = set()
for _, uid_structure_names in structure_names_by_structure_set_uid.items():
    for structure_name in uid_structure_names:
        structure_names.add(structure_name)

In [ ]:
structures_to_learn = ["patient", "brain", "eye_left", "eye_right"]
filters = {
    "study_set_must_have_all_of": structures_to_learn,
    "slice_at_least_one_of": ["brain", "eye_left", "eye_right"],
    "slice_must_have": ["patient"],
    "slice_cannot_have": [],
}

structure_uids, _ = _pipeline.get_filtered_uids(filters)

In [ ]:
structure_uids = sorted(structure_uids)

In [ ]:
validation_structure_uids = structure_uids[2:5]
training_structure_uids = structure_uids[5::]

hold_out_structure_uids = structure_uids[0:2]

In [ ]:
assert len(set(validation_structure_uids).intersection(training_structure_uids)) == 0
assert len(set(hold_out_structure_uids).intersection(training_structure_uids)) == 0
assert len(set(validation_structure_uids).intersection(hold_out_structure_uids)) == 0

In [ ]:
_, validation_ct_uids = _pipeline.get_filtered_uids(
    filters, structure_uids=validation_structure_uids
)
_, training_ct_uids = _pipeline.get_filtered_uids(
    filters, structure_uids=training_structure_uids
)
_, hold_out_ct_uids = _pipeline.get_filtered_uids(
    filters, structure_uids=hold_out_structure_uids
)

In [ ]:
assert len(set(validation_ct_uids).intersection(training_ct_uids)) == 0
assert len(set(hold_out_ct_uids).intersection(training_ct_uids)) == 0
assert len(set(validation_ct_uids).intersection(hold_out_ct_uids)) == 0

In [ ]:
mask_expansion = 5

validation_dataset = _pipeline.create_dataset(
    validation_ct_uids, structures_to_learn, expansion=mask_expansion
)
training_dataset = _pipeline.create_dataset(
    training_ct_uids[0:100], structures_to_learn, expansion=mask_expansion
)

hold_out_dataset = _pipeline.create_dataset(
    hold_out_ct_uids, structures_to_learn, expansion=mask_expansion
)

In [ ]:
def scale_images(ct_uid, x_grid, y_grid, image, mask):
    image = image / 4000
    return ct_uid, x_grid, y_grid, image, mask


validation_dataset = validation_dataset.map(scale_images)
training_dataset = training_dataset.map(scale_images)
hold_out_dataset = hold_out_dataset.map(scale_images)

In [ ]:
for ct_uid, x_grid, y_grid, image, mask in validation_dataset.take(1):
    sample_image = image
    sample_mask = mask

In [ ]:
sample_image.shape

In [ ]:
# np.max(sample_mask)

In [ ]:
sample_mask.shape

In [ ]:
def diagnostic_plotting(x_grid, y_grid, input_array, output_array):
    plt.figure(figsize=(8, 6))

    x_grid = x_grid.numpy()
    y_grid = y_grid.numpy()
    input_array = input_array.numpy()[:, :, 0]

    try:
        output_array = output_array.numpy()
    except AttributeError:
        pass

    for i in range(output_array.shape[-1]):
        contours = _mask.get_contours_from_mask(x_grid, y_grid, output_array[:, :, i])

        for contour in contours:
            plt.plot(*contour.T)

    plt.axis("equal")
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    plt.pcolormesh(x_grid, y_grid, input_array, shading="nearest")
    plt.colorbar()
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
diagnostic_plotting(x_grid, y_grid, sample_image, sample_mask)

In [ ]:
def display(image, mask, prediction=None):
    diagnostic_plotting(x_grid, y_grid, image, mask)

    if prediction is None:
        try:
            prediction = model.predict(image[None, ...])[0, ...]
        except NameError:
            return

    diagnostic_plotting(x_grid, y_grid, image, prediction)


class DisplayCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        display(sample_image, sample_mask)
        plt.show()
        print("\nSample Prediction after epoch {}\n".format(epoch + 1))


display(sample_image, sample_mask)

In [ ]:
def _activation(x):
    x = tf.keras.layers.Activation("relu")(x)

    return x


def _convolution(x, number_of_filters, kernel_size=3):
    x = tf.keras.layers.Conv2D(
        number_of_filters, kernel_size, padding="same", kernel_initializer="he_normal"
    )(x)

    return x


def _conv_transpose(x, number_of_filters, kernel_size=3):
    x = tf.keras.layers.Conv2DTranspose(
        number_of_filters,
        kernel_size,
        strides=2,
        padding="same",
        kernel_initializer="he_normal",
    )(x)

    return x

In [ ]:
def encode(
    x,
    number_of_filters,
    number_of_convolutions=2,
):
    for _ in range(number_of_convolutions):
        x = _convolution(x, number_of_filters)
        x = _activation(x)
    skip = x

    x = tf.keras.layers.MaxPool2D()(x)
    x = _activation(x)

    return x, skip

In [ ]:
def decode(
    x,
    skip,
    number_of_filters,
    number_of_convolutions=2,
):
    x = _conv_transpose(x, number_of_filters)
    x = _activation(x)

    x = tf.keras.layers.concatenate([skip, x], axis=3)

    for _ in range(number_of_convolutions):
        x = _convolution(x, number_of_filters)
        x = _activation(x)

    return x

In [ ]:
mask_dims = sample_mask.shape
assert mask_dims[0] == mask_dims[1]
grid_size = int(mask_dims[1])
output_channels = int(mask_dims[-1])

In [ ]:
BATCH_SIZE = 10

In [ ]:
strategy = tf.distribute.MirroredStrategy()
print('Number of devices: {}'.format(strategy.num_replicas_in_sync))

with strategy.scope():
    inputs = tf.keras.layers.Input((grid_size, grid_size, 1))
    x = inputs
    skips = []

    for number_of_filters in [32, 64, 128]:
        x, skip = encode(x, number_of_filters)
        skips.append(skip)

    skips.reverse()

    for number_of_filters, skip in zip([256, 128, 64], skips):
        x = decode(x, skip, number_of_filters)

    x = tf.keras.layers.Conv2D(
        output_channels,
        1,
        activation="sigmoid",
        padding="same",
        kernel_initializer="he_normal",
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=x)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(),
            tf.keras.metrics.Recall(),
            tf.keras.metrics.Precision(),
        ],
    )

In [ ]:
model.summary()

In [ ]:


display(sample_image, sample_mask)

In [ ]:
def strip_extras(ct_uid, x_grid, y_grid, image, mask):
    return image, mask


def remove_uid_and_grid(dataset):
    dataset = dataset.map(strip_extras)
    return dataset


def convert_to_batch(dataset):
    dataset = dataset.batch(BATCH_SIZE)

    return dataset


def prep_for_fit(dataset):
    dataset = remove_uid_and_grid(dataset)
    dataset = convert_to_batch(dataset)
    dataset = dataset.prefetch(20)

    return dataset

In [ ]:
history = model.fit(
    prep_for_fit(training_dataset),
    epochs=20,
    validation_data=prep_for_fit(validation_dataset),
    callbacks=[DisplayCallback()],
)